## Install required Libs

In [125]:
! pip install -qU langchain langchain-community langchain-core langchain-huggingface langchain-groq langchain-text-splitters sentence-transformers faiss-cpu rank-bm25 pymupdf nltk streamlit langsmith python-dotenv pandas openpyxl tqdm


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## NLTK configured

In [126]:
import nltk

nltk.download("punkt")
nltk.download("punkt_tab")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\baddu\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\baddu\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

# Imports

In [127]:
import os
import re
import json
from pathlib import Path
from collections import Counter

import pandas as pd
from tqdm import tqdm

import nltk
from nltk.tokenize import sent_tokenize

from dotenv import load_dotenv

# Document Loading
from langchain_community.document_loaders import PyMuPDFLoader

# Chunking
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Embeddings
from langchain_huggingface import HuggingFaceEmbeddings

# Vector Store
from langchain_community.vectorstores import FAISS

# Sparse Retrieval
from langchain_community.retrievers import BM25Retriever

# LLM
from langchain_groq import ChatGroq

print("Imports loaded successfully")

Imports loaded successfully


## config

In [128]:
import os
from dotenv import load_dotenv

load_dotenv()

# =========================
# API KEYS
# =========================

GROQ_API_KEY = os.getenv("GROQ_API_KEY")

LANGCHAIN_API_KEY = os.getenv("LANGCHAIN_API_KEY")

# =========================
# MODELS
# =========================

EMBEDDING_MODEL = "BAAI/bge-large-en-v1.5"

RERANKER_MODEL = "BAAI/bge-reranker-large"

LLM_MODEL = "llama-3.3-70b-versatile"

# =========================
# CHUNKING
# =========================

CHUNK_SIZE = 1000

CHUNK_OVERLAP = 200

# =========================
# RETRIEVAL
# =========================

MMR_K = 30

MMR_FETCH_K = 60

BM25_K = 20

RERANK_TOP_K = 5

FINAL_CONTEXT_K = 3

# =========================
# REFUSAL
# =========================



REFUSAL_MESSAGE = (
    "I can only answer HR-related questions from Zyro Dynamics policy documents."
)

# =========================
# PATHS
# =========================

DOCS_PATH = "docs"

FAISS_PATH = "faiss_index"

In [129]:
RERANK_THRESHOLD = 0.30

ENSEMBLE_WEIGHTS = [0.7, 0.3]

TOP_K_RETRIEVAL = 20

TOP_K_RERANK = 5

TOP_K_CONTEXT = 3

## Loading Docs

In [130]:
DOCS_DIR = Path("docs")

pdf_files = list(DOCS_DIR.glob("*.pdf"))

print(f"Found {len(pdf_files)} PDFs")

for pdf in pdf_files:
    print(pdf.name)

Found 11 PDFs
00_Company_Profile.pdf
01_Employee_Handbook.pdf
02_Leave_Policy.pdf
03_Work_From_Home_Policy.pdf
04_Code_of_Conduct.pdf
05_Performance_Review_Policy.pdf
06_Compensation_and_Benefits_Policy.pdf
07_IT_and_Data_Security_Policy.pdf
08_Prevention_of_Sexual_Harassment_Policy.pdf
09_Onboarding_and_Separation_Policy.pdf
10_Travel_and_Expense_Policy.pdf


In [131]:
all_docs = []

for pdf_file in pdf_files:

    loader = PyMuPDFLoader(str(pdf_file))
    docs = loader.load()

    all_docs.extend(docs)

print(f"Loaded {len(all_docs)} pages")

Loaded 39 pages


## Clean Raw Pages

In [132]:
COMMON_LINES = {
    "Zyro Dynamics Pvt. Ltd.",
    "Confidential — For Internal Use Only",
    "Navigate the Future",
    "Document Code",
    "Version",
    "Effective Date",
    "Document Owner",
    "Company Profile",
    "Employee Handbook",
    "Leave Policy",
    "Performance Review Policy",
    "Compensation and Benefits Policy",
    "Onboarding and Separation Policy",
    "HR",
    "Corporate Communications",
    "Human Resources"
}

def clean_page_text(text):

    patterns = [
        r"Page\s+\d+",
        r"Doc Code:\s*[A-Z0-9\-]+",
        r"ZDL-[A-Z]+-\d+",
        r"V\.\d+",
        r"\d{2}\s+[A-Za-z]+\s+\d{4}",
    ]

    for pattern in patterns:
        text = re.sub(
            pattern,
            "",
            text,
            flags=re.IGNORECASE
        )

    lines = []

    for line in text.split("\n"):

        line = line.strip()

        if not line:
            continue

        if line in COMMON_LINES:
            continue

        lines.append(line)

    text = "\n".join(lines)

    text = re.sub(r"\n{2,}", "\n", text)

    return text.strip()



for doc in all_docs:
    doc.page_content = clean_page_text(
        doc.page_content
    )

## Metadata

In [133]:
from pathlib import Path

for doc in all_docs:

    filename = Path(
        doc.metadata["source"]
    ).name

    policy_name = (
        Path(filename)
        .stem
        .split("_", 1)[1]
        .replace("_", " ")
    )

    doc.metadata["policy_name"] = policy_name
    doc.metadata["doc_id"] = filename
    doc.metadata["page_number"] = (
        doc.metadata["page"] + 1
    )

## Section Detection

In [134]:
import re

def is_doc_code(text):

    return bool(
        re.match(
            r'^[A-Z]{2,}-[A-Z]{2,}-\d+$',
            text.strip()
        )
    )


def detect_section(text):

    for line in text.split("\n"):

        line = line.strip()

        if not line:
            continue

        if is_doc_code(line):
            continue

        if re.match(r'^V\.\d+$', line):
            continue

        if (
            line.isupper()
            and 5 <= len(line) <= 60
            and len(line.split()) <= 8
        ):
            return line

    return "Unknown"


for doc in all_docs:

    doc.metadata["section"] = (
        detect_section(
            doc.page_content
        )
    )

## Chunking

In [135]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def section_aware_chunking(docs):

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=800,
        chunk_overlap=150,
        separators=[
            "\n\n",
            "\n",
            ". ",
            " "
        ]
    )

    final_docs = []

    for doc in docs:

        chunks = splitter.split_text(
            doc.page_content
        )

        for chunk in chunks:

            new_doc = doc.model_copy(deep=True)
            new_doc.page_content = chunk

            final_docs.append(new_doc)

    return final_docs

## Generate Chunks

In [136]:
chunks = section_aware_chunking(
    all_docs
)

print(
    f"Chunks: {len(chunks)}"
)

Chunks: 105


## Chunk IDs

In [137]:
for idx, chunk in enumerate(chunks):

    chunk.metadata["chunk_id"] = idx

## Metadata Injection

In [138]:
for chunk in chunks:

    metadata_text = f"""
Policy: {chunk.metadata.get('policy_name','')}
Section: {chunk.metadata.get('section','')}
Page: {chunk.metadata.get('page_number','')}
Document: {chunk.metadata.get('doc_id','')}
"""

    chunk.page_content = (
        metadata_text.strip()
        + "\n\n"
        + chunk.page_content
    )

## Verification

In [139]:
print("Pages :", len(all_docs))
print("Chunks:", len(chunks))

print()

print(chunks[0].metadata)

print()

print(
    chunks[0].page_content[:800]
)


print(chunks[0].metadata)
print(chunks[0].page_content[:800])

Pages : 39
Chunks: 105

{'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-05-20T11:14:29+00:00', 'source': 'docs\\00_Company_Profile.pdf', 'file_path': 'docs\\00_Company_Profile.pdf', 'total_pages': 4, 'format': 'PDF 1.4', 'title': '(anonymous)', 'author': '(anonymous)', 'subject': '(unspecified)', 'keywords': '', 'moddate': '2026-05-20T11:14:29+00:00', 'trapped': '', 'modDate': "D:20260520111429+00'00'", 'creationDate': "D:20260520111429+00'00'", 'page': 0, 'policy_name': 'Company Profile', 'doc_id': '00_Company_Profile.pdf', 'page_number': 1, 'section': 'COMPANY OVERVIEW', 'chunk_id': 0}

Policy: Company Profile
Section: COMPANY OVERVIEW
Page: 1
Document: 00_Company_Profile.pdf

COMPANY OVERVIEW
Zyro Dynamics Pvt. Ltd. is a product-based enterprise software company headquartered in the United States.
Founded in 2014 by Lamine Yamal and Aitana Bonmati, the company was built on a
single conviction: that organisations of all sizes des

## Build Embedding Model

In [140]:


embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL,
    model_kwargs={
        "device": "cpu"
    },
    encode_kwargs={
        "normalize_embeddings": True,
        "batch_size": 32
    }
)

print("Embedding model loaded")

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 3232.98it/s]


Embedding model loaded


In [141]:
test_embedding = embeddings.embed_query(
    "How many annual leaves are employees entitled to?"
)

print(type(test_embedding))
print(len(test_embedding))

<class 'list'>
1024


## Build FAISS Index

In [148]:
import langchain
import langchain_core

print(langchain.__version__)
print(langchain_core.__version__)

1.3.9
1.4.7


In [149]:
from langchain_core.stores import InMemoryStore
parent_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,
    chunk_overlap=200
)

child_splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=100
)

store = InMemoryStore()


In [ ]:
vectorstore = FAISS.from_documents(
    documents=chunks,
    embedding=embeddings
)

print("FAISS index created")
print(
    f"Indexed Chunks: {len(chunks)}"
)

In [ ]:
FAISS_PATH = "faiss_index"

vectorstore.save_local(
    FAISS_PATH
)

print(
    f"Saved to {FAISS_PATH}"
)

Saved to faiss_index


## MMR retriver

In [ ]:
mmr_retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={
    "k":20,
    "fetch_k":40
}
)

print("MMR Retriever Ready")

MMR Retriever Ready


In [ ]:
bm25_retriever = BM25Retriever.from_documents(
    chunks
)

bm25_retriever.k = 15

print("BM25 Ready")

BM25 Ready


## Ensemble Retriever

In [ ]:
from langchain_classic.retrievers import EnsembleRetriever

mmr_retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 20,
        "fetch_k": 40
    }
)

ensemble_retriever = EnsembleRetriever(
    retrievers=[
        mmr_retriever,
        bm25_retriever
    ],
    weights=[0.7, 0.3]
)

Ensemble Ready


## Reranker   

In [ ]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder(
    "BAAI/bge-reranker-large"
)

print("Reranker Loaded")

Loading weights: 100%|██████████| 393/393 [00:00<00:00, 1480.70it/s]


Reranker Loaded


In [ ]:
def rerank_documents(
    query,
    docs,
    top_k=10
):

    pairs = [
        (query, doc.page_content)
        for doc in docs
    ]

    scores = reranker.predict(pairs)

    ranked = sorted(
        zip(docs, scores),
        key=lambda x: x[1],
        reverse=True
    )

    return ranked[:top_k]

In [ ]:


def retrieve_context(query):

    docs = ensemble_retriever.invoke(query)

    ranked_docs = rerank_documents(
        query,
        docs,
        top_k=5
    )

    return [
        doc
        for doc, score in ranked_docs[:3]
    ]

In [ ]:
queries = [
    "How many sick leaves are allowed?",
    "What is maternity leave?",
    "What expenses are reimbursable?",
    "How often are reviews conducted?",
    "Password requirements?"
]

for q in queries:

    docs = retrieve_context(q)

    print("\n" + "=" * 80)
    print(q)

    print(
        docs[0].metadata["policy_name"]
    )

    print(
        docs[0].metadata["section"]
    )


How many sick leaves are allowed?
Leave Policy
ANNUAL LEAVE ENTITLEMENT

What is maternity leave?
Leave Policy
MATERNITY AND PATERNITY LEAVE

What expenses are reimbursable?
Travel and Expense Policy
ELIGIBLE AND NON-ELIGIBLE EXPENSES

How often are reviews conducted?
Performance Review Policy
REVIEW CYCLES

Password requirements?
IT and Data Security Policy
PASSWORD AND ACCESS MANAGEMENT


In [ ]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model=LLM_MODEL,
    temperature=0,
    api_key=GROQ_API_KEY
)

print("LLM Ready")

LLM Ready


In [ ]:
response = llm.invoke("Say hello in one sentence.")

print(response.content)

Hello, it's nice to meet you and I'm here to help with any questions or topics you'd like to discuss.


In [ ]:
SYSTEM_PROMPT = """
You are Zyro Dynamics HR Assistant.

Answer ONLY using the retrieved context.

Do not use outside knowledge.

Do not guess.

Do not infer.

If the answer is not explicitly present in the context, respond exactly:

"I can only answer HR-related questions from Zyro Dynamics policy documents."

Use exact policy wording whenever possible.

Use exact numbers from the context.

Do not paraphrase numerical values.

Always mention the policy name if available.
"""

In [ ]:
def format_context(docs):

    context_parts = []

    for doc in docs:

        context_parts.append(
            f"""
Policy: {doc.metadata['policy_name']}
Section: {doc.metadata['section']}
Page: {doc.metadata['page_number']}

{doc.page_content}
"""
        )

    return "\n\n".join(context_parts)

In [ ]:
def rerank_documents(
    query,
    docs,
    top_k=10
):
    pairs = [
        (query, doc.page_content)
        for doc in docs
    ]

    scores = reranker.predict(pairs)

    ranked = sorted(
        zip(docs, scores),
        key=lambda x: x[1],
        reverse=True
    )

    return ranked[:top_k]

In [ ]:
REFUSAL_MESSAGE = (
    "I can only answer HR-related questions "
    "from Zyro Dynamics policy documents."
)



In [ ]:
def debug_query(query):

    docs = ensemble_retriever.invoke(query)

    ranked_docs = rerank_documents(
        query,
        docs,
        top_k=10
    )
    
    best_doc, best_score = ranked_docs[0]
    if best_score < 0.30:
        return "I could not find sufficient information."

    print("\n")
    print("=" * 80)
    print(query)
    print(f"Score   : {best_score:.6f}")
    print(f"Policy  : {best_doc.metadata['policy_name']}")
    print(f"Section : {best_doc.metadata['section']}")

In [ ]:
def answer_question(query):

    docs = ensemble_retriever.invoke(query)

    ranked_docs = rerank_documents(
        query,
        docs,
        top_k=5
    )

    if not ranked_docs:
        return REFUSAL_MESSAGE

    best_doc, best_score = ranked_docs[0]

    if best_score < RERANK_THRESHOLD:
        return REFUSAL_MESSAGE

    final_docs = [
        doc
        for doc, score in ranked_docs[:3]
    ]

    context = format_context(final_docs)

    prompt = f"""
{SYSTEM_PROMPT}

Context:
{context}

Question:
{query}

Answer:
"""

    response = llm.invoke(prompt)

    answer = response.content

    answer += "\n\nSources:\n"

    seen = set()

    for doc in final_docs:

        source = (
            f"{doc.metadata['policy_name']} "
            f"(Page {doc.metadata['page_number']})"
        )

        if source not in seen:
            answer += f"- {source}\n"
            seen.add(source)

    return answer

In [ ]:
vectorstore.save_local(
    "faiss_index"
)

print("FAISS Saved")

FAISS Saved


In [ ]:
docs = retrieve_context(
    "How many sick leaves are allowed?"
)

for i, doc in enumerate(docs):

    print("=" * 80)
    print(i+1)

    print(doc.metadata)

    print(doc.page_content[:1500])

1
{'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-05-20T10:58:03+00:00', 'source': 'docs\\02_Leave_Policy.pdf', 'file_path': 'docs\\02_Leave_Policy.pdf', 'total_pages': 4, 'format': 'PDF 1.4', 'title': '(anonymous)', 'author': '(anonymous)', 'subject': '(unspecified)', 'keywords': '', 'moddate': '2026-05-20T10:58:03+00:00', 'trapped': '', 'modDate': "D:20260520105803+00'00'", 'creationDate': "D:20260520105803+00'00'", 'page': 1, 'policy_name': 'Leave Policy', 'doc_id': '02_Leave_Policy.pdf', 'page_number': 2, 'section': 'ANNUAL LEAVE ENTITLEMENT', 'chunk_id': 18}
Policy: Leave Policy
Section: ANNUAL LEAVE ENTITLEMENT
Page: 2

• For employees joining or separating during the leave cycle, all leave will be calculated on a pro-rata basis. • Casual Leave and Sick Leave cannot be carried forward to the next financial year, nor can they be
encashed. Any balance remaining at the end of the financial year (31 March) will lapse automaticall

In [ ]:
gold_dataset = [

    {
        "question": "How many sick leaves are allowed?",
        "answer_contains": ["10 days", "Sick Leave"]
    },

    {
        "question": "How many casual leaves are allowed?",
        "answer_contains": ["8 days", "Casual Leave"]
    },

    {
        "question": "What is maternity leave?",
        "answer_contains": ["26 weeks"]
    },

    {
        "question": "Can employees work from home?",
        "answer_contains": ["L3"]
    },

    {
        "question": "What are password requirements?",
        "answer_contains": ["12 characters"]
    },

]

In [ ]:
from sentence_transformers import SentenceTransformer
from sentence_transformers.util import cos_sim

eval_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

answer = answer_question("How many sick leaves are allowed?")
ground_truth = "Employees are entitled to 10 days of paid Sick Leave per financial year."

similarity = cos_sim(
    eval_model.encode(answer),
    eval_model.encode(
        ground_truth
    )
).item()


In [ ]:
def evaluate():

    total = len(gold_dataset)

    correct = 0

    for item in gold_dataset:

        response = answer_question(
            item["question"]
        )

        hit = all(
            phrase.lower() in response.lower()
            for phrase in item["answer_contains"]
        )

        if hit:
            correct += 1

        print("\n")
        print("=" * 80)

        print(item["question"])

        print("PASS" if hit else "FAIL")

    print("\n")

    print(
        f"Accuracy: {correct}/{total}"
    )


In [ ]:
docs = retrieve_context(
    "How many sick leaves are allowed?"
)

for doc in docs:

    print(doc.page_content[:1500])

Policy: Leave Policy
Section: ANNUAL LEAVE ENTITLEMENT
Page: 2

• For employees joining or separating during the leave cycle, all leave will be calculated on a pro-rata basis. • Casual Leave and Sick Leave cannot be carried forward to the next financial year, nor can they be
encashed. Any balance remaining at the end of the financial year (31 March) will lapse automatically. • Employees are not permitted to avail more than 2 consecutive days of Casual Leave. If additional days are
required, they must be clubbed with Earned Leave. • Sick Leave taken for more than 2 consecutive days requires a Medical Certificate from a registered medical
practitioner, to be submitted within 3 working days of returning to work. • Leave cannot be availed during the notice period unless specifically approved by the reporting manager and
HR. EARNED LEAVE (EL)
Eligibility and Accrual
Earned Leave is accrued based on the length of continuous service.
Policy: Leave Policy
Section: MATERNITY AND PATERNITY LEAVE

In [ ]:
test_questions = [

    # DIRECT FACT LOOKUP
    "How many annual leave days are employees entitled to?",
    "How many sick leave days are available?",
    "What is the maternity leave policy?",
    "What is the paternity leave policy?",
    "How long is the probation period?",
    "What is the notice period for resignation?",
    "What is the company's work from home policy?",
    "Who approves leave requests?",
    "What are office working hours?",
    "What is the reimbursement policy?",

    # PARAPHRASING
    "How many leaves do I get every year?",
    "Can I take paid leave?",
    "How many vacation days are allowed?",
    "How much annual vacation can an employee use?",
    "How many days off am I eligible for?",

    # MULTI-HOP
    "Can unused annual leave be carried forward and encashed?",
    "What happens if I resign during probation?",
    "Can a probationary employee apply for annual leave?",
    "What approvals are needed before reimbursement is processed?",
    "What happens if leave balance becomes negative?",

    # POLICY INTERPRETATION
    "Can I work remotely while on probation?",
    "Can I carry forward unused sick leave?",
    "Can leave be transferred to another employee?",
    "Can reimbursement claims be submitted late?",
    "Can I take leave during my notice period?",

    # EDGE CASES
    "What happens if a public holiday falls during annual leave?",
    "What if I get sick while on vacation?",
    "Can maternity leave and annual leave be combined?",
    "Can reimbursement be claimed without receipts?",
    "What happens if approval is not obtained?",

    # SECTION RETRIEVAL TEST
    "Explain the leave policy.",
    "Explain the reimbursement process.",
    "Explain the probation policy.",
    "Explain the employee benefits policy.",
    "Explain the attendance policy.",

    # HARD RETRIEVAL
    "What is the maximum leave carry-forward limit?",
    "Who is the final authority for leave approval?",
    "Under what circumstances can leave be denied?",
    "What documents are required for reimbursement?",
    "What are the consequences of policy violations?",

    # AMBIGUOUS QUESTIONS
    "What benefits do employees receive?",
    "What support is available for employees?",
    "How does the company handle absences?",
    "What happens after joining the company?",
    "What rights do employees have?",

    # OUT-OF-SCOPE
    "What is the capital of France?",
    "Who won the FIFA World Cup 2022?",
    "How does Bitcoin mining work?",
    "Write Python code for bubble sort.",
    "What is quantum computing?",

    # HALLUCINATION TESTS
    "Does the company provide free gym memberships?",
    "Does the company provide stock options?",
    "Does the company offer unlimited leave?",
    "Does the company provide free housing?",
    "Does the company provide company cars?"
]

In [ ]:
for i, question in enumerate(test_questions, 1):

    print("=" * 100)
    print(f"TEST {i}")
    print(f"QUESTION: {question}")

    try:

        answer = answer_question(question)

        print("\nANSWER:")
        print(answer)

    except Exception as e:

        print(f"ERROR: {e}")

    print("\n")

In [ ]:
for question in test_questions[:10]:

    print("="*100)
    print(question)

    docs = retrieve_context(question)

    for i, doc in enumerate(docs[:3]):

        print(f"\nContext {i+1}")

        print(
            doc.page_content[:500]
        )

In [ ]:
!pip install -U langsmith


In [ ]:
from langsmith import traceable

@traceable(
    name="hybrid_retrieval"
)
def retrieve_context(
    query: str,
    top_k: int = 20
):

    docs = ensemble_retriever.invoke(query)

    return docs

In [ ]:
@traceable(
    name="reranker_stage"
)
def rerank_documents(
    query: str,
    docs,
    top_k: int = 5
):

    pairs = [
        (query, doc.page_content)
        for doc in docs
    ]

    scores = reranker.predict(pairs)

    scored_docs = sorted(
        zip(docs, scores),
        key=lambda x: x[1],
        reverse=True
    )

    return scored_docs[:top_k]


In [ ]:
REFUSAL_MESSAGE = (
    "I can only answer HR-related questions "
    "from Zyro Dynamics policy documents."
)

RERANK_THRESHOLD = 0.30


@traceable(
    name="answer_question"
)
def answer_question(query: str):

    docs = retrieve_context(
        query=query,
        top_k=20
    )

    ranked_docs = rerank_documents(
        query=query,
        docs=docs,
        top_k=5
    )

    if not ranked_docs:
        return REFUSAL_MESSAGE

    best_doc, best_score = ranked_docs[0]

    if best_score < RERANK_THRESHOLD:
        return REFUSAL_MESSAGE

    final_docs = [
        doc
        for doc, score in ranked_docs[:3]
    ]

    context = "\n\n".join(
        doc.page_content
        for doc in final_docs
    )

    prompt = f"""
You are Zyro Dynamics HR Assistant.

Answer ONLY using the retrieved context.

Do not use outside knowledge.

Do not guess.

Do not infer.

If the answer is not explicitly present
in the context respond exactly:

"{REFUSAL_MESSAGE}"

Use exact policy wording whenever possible.

Use exact numbers.

Use exact leave names.

Always mention policy names.

Context:
{context}

Question:
{query}

Answer:
"""

    response = llm.invoke(prompt)

    answer = response.content

    answer += "\n\nSources:\n"

    seen = set()

    for doc in final_docs:

        source = (
            f"{doc.metadata['policy_name']} "
            f"(Page {doc.metadata['page_number']})"
        )

        if source not in seen:
            answer += f"- {source}\n"
            seen.add(source)

    return answer

In [ ]:
@traceable(
    name="evaluate_rag"
)
def evaluate(
    test_questions
):

    results = []

    for item in test_questions:

        answer = answer_question(
            item["question"]
        )

        results.append(
            {
                "question": item["question"],
                "answer": answer
            }
        )

    return results